# V2 push — SFT v2 seeds 42 + 1234 (the confirmation runs)

**Runtime: A100 (~40 min/seed) or L4 (~1h20/seed).** Checkpointed per seed.

Seed 3407 hit **34.51%** on the held-out exam — 4.1 over the OctoCoder-16B
target. But v1's exam cross-seed sd was 4.2, so a single seed cannot carry the
claim. This notebook trains the SAME v2 recipe (data v1 mixture, 2 epochs,
ctx 1280, r32/α64) with seeds 42 and 1234 — **ep2 is the pre-committed
checkpoint** (chosen on the tracer BEFORE these seeds run).

Per seed: train ep1 → save → train ep2 → save → quick sanity eval on the SAME
60 new-dev bugs the tracer used (k=8; tracer ep2 scored 73.5/91.7 there).
Exam runs happen in notebook 16 (L4).

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
v1_bugs = json.load(open(f'{PHASE8}/data_v1_bugs.json'))
v1_restraint = json.load(open(f'{PHASE8}/data_v1_restraint.json'))
train_v1 = [b for b in v1_bugs if b['split'] == 'train']
clean_train = [r for r in v1_restraint if r['split'] == 'train']
dev_new = [b for b in v1_bugs if b['split'] == 'dev' and b.get('gen') == 'deepseek_v1']
dev_new_sample = random.Random(3407).sample(dev_new, 60)   # SAME 60 as tracer/13b
print(len(train_v1), 'train |', len(clean_train), 'restraint |', len(dev_new_sample), 'sanity dev')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Helpers (grading + quick new-dev sanity eval)
import subprocess, tempfile, torch
from concurrent.futures import ThreadPoolExecutor
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

def sanity_eval(model, tok, tag, k=8):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_new_sample:
        text = tok.apply_chat_template(
            [{'role': 'user', 'content': build_repair_prompt(b['buggy'], b['test_list'])}],
            tokenize=False, add_generation_prompt=True)
        enc = tok([text], return_tensors='pt', padding=True, padding_side='left').to('cuda')
        with torch.no_grad():
            out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                                 num_return_sequences=k, max_new_tokens=512,
                                 pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        codes = [extract_code(t) for t in gen]
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    pk = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    json.dump({'tag': tag, 'pass@1': p1, 'pass@8': pk, 'per_bug': per_bug},
              open(f'{PHASE8}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@8={pk*100:.1f}%   (tracer ep2: 73.5/91.7)")

In [ ]:
# Train per seed: exact tracer recipe, 2 epochs, save both, sanity-eval ep2
def run_seed(seed):
    print(f'===== SFT v2 SEED {seed} =====')
    rng = random.Random(seed)
    mixture = [{'prompt': build_repair_prompt(b['buggy'], b['test_list']),
                'answer': '```python\n' + b['fixed'].strip() + '\n```'} for b in train_v1]
    mixture += [{'prompt': build_repair_prompt(r['code'], r['test_list']),
                 'answer': '```python\n' + r['code'].strip() + '\n```'} for r in clean_train]
    rng.shuffle(mixture)
    model, tok = FastLanguageModel.from_pretrained(
        'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1280,
        load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=64, lora_dropout=0,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=seed)
    def to_text(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['answer']}]
        return {'text': tok.apply_chat_template(msgs, tokenize=False)}
    ds = Dataset.from_list(mixture).map(to_text)
    def make_trainer():
        FastLanguageModel.for_training(model)
        t = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
            dataset_text_field='text',
            args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
                num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
                warmup_ratio=0.05, logging_steps=20, seed=seed, output_dir='/content/out',
                report_to='none', save_strategy='no'))
        return train_on_responses_only(t,
            instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
    make_trainer().train()
    model.save_pretrained(f'{PHASE8}/sft_v2_s{seed}_ep1')
    make_trainer().train()
    model.save_pretrained(f'{PHASE8}/sft_v2_s{seed}_ep2')
    sanity_eval(model, tok, f'sftv2_ep2_newdev_seed{seed}')
    del model, tok
    gc.collect(); torch.cuda.empty_cache()

for s in (42, 1234):
    if os.path.exists(f'{PHASE8}/dev_eval_sftv2_ep2_newdev_seed{s}.json'):
        print(f'seed {s} already done, skipping'); continue
    run_seed(s)
print('\nDone. Exam runs: notebook 16 (L4).')

## Bring back to the session
The two **sanity-eval lines** (vs tracer's 73.5/91.7). Then run notebook 16
on an L4 for the exam confirmation.